# Bayesian Unfolding

This notebook contains the traditional **Bayesian unfolding** of the smeared data.

At first, we need to import ROOT with the RooUnfold library, and select data samples for training (creating the response matrix and unfolding matrix) and unfolding.

In [ ]:
import ROOT
ROOT.gSystem.Load("libRooUnfold") # load RooUnfold
ROOT.gStyle.SetOptStat(0) # disables the stat boxes in plots

# select data for training and unfolding by their seed
seedTraining = 67
seedUnfolding = 420

### 1D Distributions

We load the histograms generated in the plot-ea.ipynb notebook.

In [ ]:
# open files for reading
fileTraining = ROOT.TFile(f"data/{seedTraining}/events{seedTraining}_plots.root", "READ")
fileUnfolding = ROOT.TFile(f"data/{seedUnfolding}/events{seedUnfolding}_plots.root", "READ") # a different simulation -- represents the measured data

distributions = {'Nch': 'Multiplicity', 'S0pT1': 'Unweighted Spherocity'} # dictionary of distributions we want to unfold (e.g., multiplicity, spherocity, pT spectra, ...), syntax: abbreviation: full name
histos = {} # create an empty dictionary where we will store our histograms

# fill the distributions dictionary
for abbrev, name in distributions.items():
    # syntax: abbreviation: (name, true training data, smeared training data, response matrix from training data, independent true data, smeared data to be unfolded)
    histos[abbrev] = [fileTraining.Get(f"histo{seedTraining}{abbrev}{kind}") for kind in ["True", "Smeared", "RM"]] # fill in the training data
    histos[abbrev] += [fileUnfolding.Get(f"histo{seedUnfolding}{abbrev}{kind}") for kind in ["True", "Smeared"]] # append the data to be unfolded
    histos[abbrev].insert(0, name) # prepend the full name of the observable

/home/javkovsky/anaconda3/envs/star-bachelor-env/lib/python3.11/site-packages/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /home/javkovsky/anaconda3/envs/star-bachelor-env/include/site/python3.11); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "


Afterwards, we construct response matrices compatible with RooUnfold for each observable from the training data. Then, we proceed with the Bayesian unfolding of the measured data and plot all distributions for comparison. Lastly, we perform a closure test to evaluate the quality of unfolding.

In [2]:
# loop over different observables
for abbrev, data in histos.items():
    # define histogram variables from the histos dictionary
    name = data[0]
    histoTrainingTrue = data[1]
    histoTrainingSmeared = data[2]
    histoTrainingRM = data[3]
    histoUnfoldingTrue = data[4]
    histoUnfoldingSmeared = data[5]

    # ---------
    # UNFOLDING
    # ---------

    # create a RooUnfold response matrix
    response = ROOT.RooUnfoldResponse(histoTrainingSmeared, histoTrainingTrue, histoTrainingRM)

    # Bayesian unfolding with nIter iterations
    nIter = 5 # define the number of iterations
    #unfolded = ROOT.RooUnfoldBayes(response, histoUnfoldingSmeared) # initialize the unfolding object -- SEM DAT MISTO histoUnfoldingSmeared --> histoTrainingSmeared?
    unfolded = ROOT.RooUnfoldBayes(response, histoTrainingSmeared)
    
    unfolded.SetIterations(nIter - 1) # perform nIter-1 iterations
    histoPenultimate = unfolded.Hunfold().Clone("histoPenultimate") # extract the histogram after the penultimate iteration (we must use the .Clone() method otherwise defining histoUnfolded below would overwrite this histogram)
    histoPenultimate.SetDirectory(0) # makes sure that Python takes memory management ownership of the object

    unfolded.SetIterations(nIter) # perform nIter iterations
    histoUnfolded = unfolded.Hunfold().Clone("histoUnfolded") # extract the unfolded histogram (we must once again use the .Clone() method for several reasons: .Hunfold() returns a pointer to an internal histogram managed by the RooUnfold object; if we modified (e.g., scaled) the histogram without cloning it, it would alter the internal state of the unfolding engine and if we then wanted to extract some more information about the RooUnfold object (e.g., a covariance matrix), the math would be broken)
    histoUnfolded.SetDirectory(0)

    # --------------
    # UNFOLDING PLOT
    # --------------

    # set up canvas for drawing
    canvasUnfolding = ROOT.TCanvas(f"canvasUnfolding{abbrev}", f"Bayesian Unfolding of {name}", 800, 600)
    canvasUnfolding.SetLogy() 

    # normalize the histograms
    for histo in [histoUnfoldingTrue, histoUnfoldingSmeared, histoPenultimate, histoUnfolded]:
        integral = histo.Integral()
        if integral > 0:
            histo.Scale(1.0 / integral)

    # customize the histograms (to customize title, labels, ..., we have to modify the first drawn histogram)
    histoUnfoldingTrue.SetTitle(f"Bayesian Unfolding of {name};Values;Normalized Events") # set title and labels of axes

    max_val = max(histoUnfoldingTrue.GetMaximum(), histoUnfoldingSmeared.GetMaximum(), histoPenultimate.GetMaximum(), histoUnfolded.GetMaximum())
    histoUnfoldingTrue.SetMaximum(max_val * 4.5) # scale the y-axis to prevent cut-off

    histoUnfoldingTrue.SetLineColor(ROOT.kGreen)
    histoUnfoldingSmeared.SetLineColor(ROOT.kBlue)
    histoPenultimate.SetLineColor(ROOT.kOrange)
    histoUnfolded.SetLineColor(ROOT.kRed)

    histoPenultimate.SetMarkerStyle(ROOT.kOpenSquare)
    histoPenultimate.SetMarkerColor(ROOT.kOrange)
    histoPenultimate.SetMarkerSize(0.8)

    histoUnfolded.SetMarkerStyle(ROOT.kFullCircle)
    histoUnfolded.SetMarkerColor(ROOT.kRed)
    histoUnfolded.SetMarkerSize(0.5)

    # drawing
    histoUnfoldingTrue.Draw("HIST")
    histoUnfoldingSmeared.Draw("HIST SAME") # "SAME" tells ROOT to draw the histogram on top of the previous one
    histoPenultimate.Draw("PE SAME") # "P" forces a point scatter
    histoUnfolded.Draw("PE SAME") # "E" ensures that error bars are drawn

    # legend
    legend = ROOT.TLegend(0.69, 0.74, 0.89, 0.89) # set coordinates as fractions of the canvas (x1, y1, x2, y2)
    legend.SetBorderSize(0) # removes the border around the legend box
    
    legend.AddEntry(histoUnfoldingTrue, "True Data", "l")
    legend.AddEntry(histoUnfoldingSmeared, "Smeared Data", "l")
    legend.AddEntry(histoPenultimate, f"{nIter - 1}. Iteration", "p")
    legend.AddEntry(histoUnfolded, f"{nIter}. Iteration", "p")
    
    legend.Draw()

    # save
    canvasUnfolding.SaveAs(f"img/{seedUnfolding}/{seedUnfolding}{abbrev}Unfolding.pdf") # save the canvas as a PDF file

    # ------------
    # CLOSURE TEST
    # ------------

    # define the histogram of the unfolded data and true data ratio
    histoRatio = histoUnfolded.Clone(f"closureRatio{abbrev}") # set the numerator = unfolded data of the ratio
    histoRatio.SetDirectory(0)
    histoRatio.Divide(histoUnfoldingTrue) # divide by the true data histogram

    # set up canvas
    canvasClosure = ROOT.TCanvas(f"canvasClosure{abbrev}", f"losure Test for Unfolding of {name}", 800, 600)

    # customize the histogram
    histoRatio.SetTitle(f"Closure Test for Unfolding of {name};Values;Unfolded / True Data")

    histoRatio.SetMinimum(0.7)
    histoRatio.SetMaximum(1.3)

    histoRatio.SetLineColor(ROOT.kPink)
    histoRatio.SetMarkerStyle(ROOT.kFullCircle)
    histoRatio.SetMarkerColor(ROOT.kPink)
    histoRatio.SetMarkerSize(0.5)

    # draw the histogram
    histoRatio.Draw("PE") 
    
    # draw the reference line at y=1
    xMin = histoRatio.GetXaxis().GetXmin() # extract x-axis limits
    xMax = histoRatio.GetXaxis().GetXmax()
    line = ROOT.TLine(xMin, 1.0, xMax, 1.0) # create a line between xMin, xMax at y=1 (TLine(x1, y1, x2, y2))
    line.SetLineColor(ROOT.kGreen)
    line.SetLineStyle(2) # dashed line
    line.SetLineWidth(2)
    line.Draw("SAME")

    # save
    canvasClosure.SaveAs(f"img/{seedUnfolding}/{seedUnfolding}{abbrev}ClosureTest.pdf")

Using response matrix priors
Priors:

Vector (70)  is as follows

     |        1  |
------------------
   0 |0.00022 
   1 |0.00062 
   2 |0.00199 
   3 |0.00354 
   4 |0.00548 
   5 |0.00835 
   6 |0.01165 
   7 |0.01505 
   8 |0.01922 
   9 |0.02298 
  10 |0.02756 
  11 |0.03234 
  12 |0.0359 
  13 |0.03988 
  14 |0.04229 
  15 |0.04489 
  16 |0.04618 
  17 |0.04794 
  18 |0.04714 
  19 |0.04751 
  20 |0.04681 
  21 |0.04574 
  22 |0.04286 
  23 |0.03994 
  24 |0.03698 
  25 |0.03506 
  26 |0.0322 
  27 |0.02912 
  28 |0.0268 
  29 |0.02307 
  30 |0.02083 
  31 |0.01907 
  32 |0.01604 
  33 |0.01359 
  34 |0.01139 
  35 |0.0097 
  36 |0.00943 
  37 |0.00741 
  38 |0.00604 
  39 |0.00479 
  40 |0.00435 
  41 |0.00344 
  42 |0.00311 
  43 |0.00242 
  44 |0.00203 
  45 |0.00143 
  46 |0.00123 
  47 |0.00103 
  48 |0.00062 
  49 |0.00069 
  50 |0.0004 
  51 |0.00038 
  52 |0.00025 
  53 |0.00028 
  54 |0.00021 
  55 |0.00019 
  56 |9e-05 
  57 |7e-05 
  58 |6e-05 
  59 |2e-05 
  60 |2e-

Info in <TCanvas::Print>: pdf file img/420/420NchUnfolding.pdf has been created
Info in <TCanvas::Print>: pdf file img/420/420NchClosureTest.pdf has been created
Info in <TCanvas::Print>: pdf file img/420/420S0pT1Unfolding.pdf has been created
Info in <TCanvas::Print>: pdf file img/420/420S0pT1ClosureTest.pdf has been created


### $p_\mathrm{T}$ Spectra

In the end, we close the files we opened in the beginning.

In [3]:
fileTraining.Close()
fileUnfolding.Close()